In [1]:
from langgraph.graph import StateGraph, START , END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()
import os


In [2]:
api_key = os.getenv("OPENAI_API_KEY")

In [3]:
# Agent 1 - GPT-4o Mini (via OpenRouter)
openai_llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0.7,
    api_key = os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

# Agent 2 - Llama 3 (via OpenRouter)
huggingface_llm = ChatOpenAI(
    model="meta-llama/llama-3.3-70b-instruct",
    temperature=0.7,
    api_key = os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

# Agent 3 - Qwen Coder (via OpenRouter)
groq_llm = ChatOpenAI(
    model="qwen/qwen3-coder",
    temperature=0.2,
    api_key = os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
     max_tokens=1000
)

In [4]:
class ReviewState(TypedDict, total=False):
    document: str
    grammar_feedback: str
    legal_feedback: str
    format_feedback: str
    final_review: str

In [ ]:
# agent 1: Grammar and Clarity Check
def grammar_check(state: ReviewState):
    print("Grammar check started: OpenAI grammar agent")
    response = openai_llm.invoke(f"""
    You are a Senior Grammar and Clarity Editor.

    TASK:
    Review the document for grammar, spelling, punctuation, sentence clarity, and readability.

    DOCUMENT:
    {state['document']}

    STRICT OUTPUT FORMAT:

    GRAMMAR ISSUES:
    - issue 1
    - issue 2

    CLARITY IMPROVEMENTS:
    - improvement 1
    - improvement 2

    REWRITTEN VERSION:
    (provide a cleaner version of the document with grammar and clarity fixes)
    """)

    print("Grammar check completed")
    return {"grammar_feedback": response.content.strip()}

In [ ]:

# agent 2: Legal Risk Check
def legal_check(state: ReviewState):
    print("Legal check started: Groq legal agent")
    response = groq_llm.invoke(f"""
    You are a Legal Risk Review Assistant.

    TASK:
    Review the document for possible legal, compliance, liability, privacy, and obligation risks.

    DOCUMENT:
    {state['document']}

    IMPORTANT:
    This is not legal advice. Give practical review notes only.

    STRICT OUTPUT FORMAT:

    LEGAL RISK AREAS:
    - risk 1
    - risk 2

    UNCLEAR OR MISSING TERMS:
    - term 1
    - term 2

    SUGGESTED SAFER WORDING:
    - suggestion 1
    - suggestion 2
    """)

    print("Legal check completed")
    return {"legal_feedback": response.content.strip()}

In [ ]:

# agent 3: Format Check
def format_check(state: ReviewState):
    print("Format check started: Hugging Face format agent")
    response = huggingface_llm.invoke(f"""
    You are a Professional Document Formatting Specialist.

    TASK:
    Review the document for structure, headings, spacing, list formatting, consistency, and professional presentation.

    DOCUMENT:
    {state['document']}

    STRICT OUTPUT FORMAT:

    FORMAT ISSUES:
    - issue 1
    - issue 2

    STRUCTURE RECOMMENDATIONS:
    - recommendation 1
    - recommendation 2

    FORMATTED OUTLINE:
    (show the improved structure or layout)
    """)

    print("Format check completed")
    return {"format_feedback": response.content.strip()}

In [8]:
# agent 4: Merge Review Reports
def review_merge(state: ReviewState):
    print("Review merge started: OpenAI merge agent")
    response = openai_llm.invoke(f"""
    You are a Senior Document Review Lead.

    TASK:
    Merge the three independent review reports into one final document review.

    ORIGINAL DOCUMENT:
    {state['document']}

    GRAMMAR REVIEW:
    {state['grammar_feedback']}

    LEGAL REVIEW:
    {state['legal_feedback']}

    FORMAT REVIEW:
    {state['format_feedback']}

    STRICT RULES:
    1. Combine all important findings
    2. Remove duplicate or repeated points
    3. Keep legal notes clearly separated from grammar and formatting notes
    4. Do not claim to provide official legal advice
    5. Give a clear final improved version of the document

    STRICT OUTPUT FORMAT:

    FINAL REVIEW SUMMARY:
    (short summary)

    GRAMMAR AND CLARITY FIXES:
    - point 1
    - point 2

    LEGAL AND COMPLIANCE NOTES:
    - point 1
    - point 2

    FORMAT AND STRUCTURE FIXES:
    - point 1
    - point 2

    FINAL IMPROVED DOCUMENT:
    (complete improved document)
    """)

    print("Review merge completed")
    return {"final_review": response.content.strip()}

In [9]:
graph = StateGraph(ReviewState)
graph.add_node("grammar_check", grammar_check)
graph.add_node("legal_check", legal_check)
graph.add_node("format_check", format_check)
graph.add_node("review_merge", review_merge)

In [10]:
# START three parallel review agents
graph.add_edge(START, "grammar_check")
graph.add_edge(START, "legal_check")
graph.add_edge(START, "format_check")

# Wait for all three parallel checks, then merge
graph.add_edge(["grammar_check", "legal_check", "format_check"], "review_merge")

graph.add_edge("review_merge", END)

In [11]:
app = graph.compile()

In [16]:
document_text = """
the resume is a bit rough and needs some polishing. It has some grammatical errors, unclear phrasing, and the formatting is inconsistent. Additionally, there are some legal disclaimers that need to be reviewed for compliance. Overall, it needs a comprehensive review to ensure it is professional and clear."""
result = app.invoke({"document": document_text})
print(result["final_review"])

Format check started: Hugging Face format agent
Grammar check started: OpenAI grammar agent
Legal check started: Groq legal agent
Grammar check completed
Legal check completed
Format check completed
Review merge started: OpenAI merge agent
Review merge completed
**FINAL REVIEW SUMMARY:**
The resume requires significant refinement to enhance its professionalism, including corrections for grammatical errors, improved clarity, and consistent formatting. Additionally, it is crucial to address legal disclaimers to ensure compliance with relevant regulations.

**GRAMMAR AND CLARITY FIXES:**
- Changed "the resume" to "The resume" for proper capitalization.
- Clarified "a bit rough and needs some polishing" to "requires significant refinement to enhance its professionalism."
- Specified that legal disclaimers need to be reviewed for compliance with relevant regulations.

**LEGAL AND COMPLIANCE NOTES:**
- Inconsistent formatting and unprofessional presentation may create liability concerns if u

In [17]:
print(result["grammar_feedback"])

GRAMMAR ISSUES:
- "the resume" should start with a capital letter: "The resume"
- "a bit rough" could be more formally expressed.

CLARITY IMPROVEMENTS:
- "a bit rough and needs some polishing" could be more specific about what needs polishing.
- "some legal disclaimers that need to be reviewed for compliance" could specify what compliance pertains to.

REWRITTEN VERSION:
The resume requires significant refinement to enhance its professionalism. It contains several grammatical errors and unclear phrases, and the formatting is inconsistent. Additionally, the legal disclaimers need to be reviewed to ensure compliance with relevant regulations. Overall, a comprehensive review is necessary to ensure the document is both professional and clear.


In [18]:
print(result["legal_feedback"])

LEGAL RISK AREAS:
- Inconsistent formatting and unprofessional presentation may create liability concerns if used for employment purposes
- Missing or inadequate legal disclaimers could expose to compliance risks

UNCLEAR OR MISSING TERMS:
- Specific grammatical errors referenced but not identified
- "Legal disclaimers" - type and scope not specified
- Professional standards expected - not defined

SUGGESTED SAFER WORDING:
- "Resume requires proofreading and formatting standardization to meet professional standards"
- "Document contains areas requiring grammatical correction and structural improvement"
- "Legal disclaimers should be reviewed by qualified counsel before use"
- "This document is provided as-is without warranty or guarantee of accuracy"


In [19]:
print(result["format_feedback"])

FORMAT ISSUES:
- Inconsistent spacing between sections and paragraphs
- Lack of clear headings and subheadings to differentiate between sections
- Grammatical errors and unclear phrasing throughout the document
- Inconsistent font and font size used throughout the document
- No clear separation between different types of information (e.g. work experience, education, skills)

STRUCTURE RECOMMENDATIONS:
- Use a standard font (e.g. Arial, Calibri, or Helvetica) and font size (e.g. 10 or 11 points) throughout the document
- Organize the content into clear sections with headings (e.g. Work Experience, Education, Skills) and subheadings (e.g. Job Title, Company Name)
- Use bullet points and white space to make the content more readable and easy to scan
- Consider adding a summary or objective statement at the beginning of the document to provide an overview of the candidate's experience and qualifications

FORMATTED OUTLINE:
I. Contact Information
    * Name
    * Address
    * Phone Number


In [20]:
print(result["final_review"])

**FINAL REVIEW SUMMARY:**
The resume requires significant refinement to enhance its professionalism, including corrections for grammatical errors, improved clarity, and consistent formatting. Additionally, it is crucial to address legal disclaimers to ensure compliance with relevant regulations.

**GRAMMAR AND CLARITY FIXES:**
- Changed "the resume" to "The resume" for proper capitalization.
- Clarified "a bit rough and needs some polishing" to "requires significant refinement to enhance its professionalism."
- Specified that legal disclaimers need to be reviewed for compliance with relevant regulations.

**LEGAL AND COMPLIANCE NOTES:**
- Inconsistent formatting and unprofessional presentation may create liability concerns if used for employment purposes.
- Legal disclaimers should be reviewed by qualified counsel before use to mitigate compliance risks.

**FORMAT AND STRUCTURE FIXES:**
- Standardized font to Arial and size to 11 points for uniformity.
- Organized content into clearly 

In [21]:
print(app.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	grammar_check(grammar_check)
	legal_check(legal_check)
	format_check(format_check)
	review_merge(review_merge)
	__end__([<p>__end__</p>]):::last
	__start__ --> format_check;
	__start__ --> grammar_check;
	__start__ --> legal_check;
	format_check --> review_merge;
	grammar_check --> review_merge;
	legal_check --> review_merge;
	review_merge --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

